# 02 - From listens to genres

**Goal of this notebook:** find a genre for every song that was played in one year, and count how many listens each genre got.

The listens have no genre, so we go in steps:

1. **Count the songs** - turn the listens into a list of distinct songs, each with the number of listens (DuckDB, straight from the parquet files). Listens that the extract flags as duplicates (`is_dup`) are left out.
2. **Match keys** - make keys from artist + title (with cleaning rules for `feat.`, brackets, ` - Remastered`, ... and for collaborations).
3. **Find the songs in MusicBrainz** - compare the keys with MusicBrainz's canonical table to get the MusicBrainz ids.
4. **Find the genres** - MusicBrainz users tag songs with genres. We take the most-voted genre.
5. **Save** the song table (parquet) and the genre counts (csv).

Everything is controlled by `YEAR` and `DEV_ONLY` below. Change `YEAR` for another year; set `DEV_ONLY = False` for all users. To run many years in a row, use `02.02_run_all_years.py` (it sets `YEAR` for you).

## 1. Setup

In [147]:
%pip install -q duckdb zstandard pyarrow unidecode

Note: you may need to restart the kernel to use updated packages.


In [148]:
from pathlib import Path
import gc, os, io, re, tarfile, time
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as pacsv
import zstandard
from unidecode import unidecode

RAW = Path("/Users/hanshornbek/Desktop/Projects/Data Mining project")
EXTRACT = RAW / "Extract"                                                    # the parquet files + the user csv files
CANON = RAW / "Raw data/MusicBrainz Canonical/musicbrainz-canonical-dump-20261103-080003.tar.zst"
MB = RAW / "Raw data/MusicBrainz Full/tables/mbdump"                        # the MusicBrainz database tables

# YEAR and DEV_ONLY can be set here by hand. If the environment variables YEAR / DEV_ONLY exist
# (02.02_run_all_years.py sets them), they win. That is how one script can run this notebook for every year.
YEAR = int(os.environ.get("YEAR", 2013))                    # the year we look at
DEV_ONLY = os.environ.get("DEV_ONLY", "1") == "1"           # True: only the development users (bucket 0-9). False: all users.
DROP_DUPS = True       # True: leave out the listens the extract flags as duplicates (is_dup = same user, same second, same song)
MEMORY_LIMIT = "3GB"   # how much memory DuckDB may use

SUFFIX = "dev" if DEV_ONLY else "all"
DATA = Path.cwd().parent / "Output_dev" / "data"
DATA.mkdir(parents=True, exist_ok=True)
PARQ = DATA / "parquets" / "songs"                                               # the song tables (parquet) go here
PARQ.mkdir(parents=True, exist_ok=True)
CSV_DIR = Path.cwd().parent / "Output_dev" / "genre_counts" / str(YEAR)          # the small csv results go here, one folder per year
CSV_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

def run(sql):
    """Run a SQL statement that creates or saves something (CREATE TABLE, COPY, PRAGMA) and finish it completely."""
    con.execute(sql).fetchall()

def scalar(sql):
    """Run a query that returns one number, and give back that number."""
    return con.execute(sql).fetchall()[0][0]

run(f"PRAGMA memory_limit='{MEMORY_LIMIT}'")
run(f"PRAGMA temp_directory='{DATA / '.tmp'}'")           # where DuckDB puts temporary files if memory runs short (git ignores this folder)
print(f"year {YEAR}, users: {'development users only' if DEV_ONLY else 'all users'}")
print("all input files found:", EXTRACT.exists() and CANON.exists() and MB.exists())

year 2013, users: development users only
all input files found: True


## 2. Count the songs

One DuckDB query goes through all parquet files of the year and does this:

- `WHERE user_id IN (...)` - keep only the listens of the development users (left out when `DEV_ONLY = False`)
- `GROUP BY artist_name, track_name` - put all listens of the same song together
- `count(*)` - how many listens each song has
- `max(recording_mbid)` etc. - if ListenBrainz gave the song a MusicBrainz id in some listen, keep one of them

`coalesce(x, '')` turns a missing name into an empty text.

In [149]:
users = pd.read_csv(EXTRACT / "cohort_users.csv")
dev_users = users.loc[users["bucket"] < 10, ["user_id"]]
con.register("dev_users", dev_users)

files = EXTRACT / "extract" / f"year={YEAR}" / "*.parquet"        # * means: all the files in the folder
user_filter = "WHERE user_id IN (SELECT user_id FROM dev_users)" if DEV_ONLY else "WHERE true"   # which users
dup_filter = "AND NOT coalesce(is_dup, false)" if DROP_DUPS else ""                                  # leave out duplicate listens

start = time.time()
songs = con.sql(f"""
    SELECT coalesce(artist_name, '')            AS artist_name,
           coalesce(track_name, '')             AS track_name,
           count(*)                             AS n_listens,
           coalesce(max(recording_mbid), '')    AS lb_recording_mbid,
           coalesce(max(artist_mbid), '')       AS lb_artist_mbid,
           coalesce(max(release_mbid), '')      AS lb_release_mbid
    FROM read_parquet('{files}')
    {user_filter} {dup_filter}
    GROUP BY 1, 2
""").df()
print(f"{len(songs):,} distinct songs, {songs['n_listens'].sum():,} listens ({time.time() - start:.0f} s)")
songs.sort_values("n_listens", ascending=False).head(8)

1,064,329 distinct songs, 4,988,913 listens (2 s)


,artist_name,track_name,n_listens,lb_recording_mbid,lb_artist_mbid,lb_release_mbid
731977,Imagine Dragons,Radioactive,2347,bd61eda3-eb77-4634-ba66-4a084f7f8455,,
83040,Imagine Dragons,Demons,1724,87dede50-097f-4064-9a9b-bd4851363971,,
683820,Daft Punk,Give Life Back to Music,1437,0f97f0da-c46f-4008-b130-a0b71232f76e,,
384958,Avicii,Wake Me Up,1293,72281f36-2845-4c70-95fd-bffbbeab1ef8,,
567303,Daft Punk,Giorgio by Moroder,1198,cac27da9-20c5-4722-9a25-f2c26e2c04cf,,
218053,Daft Punk,Get Lucky (feat. Pharrell Williams),1152,011049b6-1076-4484-a9fe-4fd4cb7466a7,,
101259,Daft Punk,The Game of Love,1151,294a1b4d-ebc0-4b03-be25-6171d382cb58,,
323423,Feisty 2 Gunns,"Empty Whitt Out You (Feisty 2 Gunns, Chino Gra...",1116,,,


**Check:** the extract has a table of how many listens each user has per year. Our listens must add up to exactly that, minus the duplicate listens we left out (we count those in the extract too).

In [150]:
year_counts = pd.read_csv(EXTRACT / "cohort_year_counts.csv")
in_year = year_counts[year_counts["year"] == YEAR]
if DEV_ONLY:
    in_year = in_year[in_year["user_id"].isin(dev_users["user_id"])]
expected = in_year["listens"].sum()
n_dups = 0
if DROP_DUPS:
    n_dups = scalar(f"SELECT count(*) FROM read_parquet('{files}') {user_filter} AND coalesce(is_dup, false)")
    expected = expected - n_dups
    print(f"duplicate listens left out: {n_dups:,}")

print(f"listens in our songs: {songs['n_listens'].sum():,}")
print(f"listens expected:     {expected:,}")
print("the same!" if songs["n_listens"].sum() == expected else "NOT the same - something is wrong")
assert songs["n_listens"].sum() == expected, "listens do not add up - stop here and look at it"

listens in our songs: 4,988,913
listens expected:     4,988,913
the same!


## 3. The match key

MusicBrainz has a table (the *canonical table*) where every song has a column `combined_lookup`. It is artist + title with everything that is not a letter or digit removed, in lowercase, with other alphabets turned into plain letters. We build our key **in exactly the same way**, so keys can be compared directly.

In [151]:
def make_key(artist, title):
    """MusicBrainz's own recipe: artist + title, only letters/digits, lowercase, then unidecode."""
    return unidecode(re.sub(r"[^\w]+", "", artist + title).lower())

print(make_key("Blümchen", "Nur Geträumt"))
print(make_key("Wu-Tang Clan", "C.R.E.A.M. (Cash Rules Everything Around Me)"))
print(repr(make_key("茶太", "環")))

blumchennurgetraumt
wutangclancreamcashruleseverythingaroundme
'Cha Tai Huan '


The last one looks odd (`Cha Tai Huan `, with capitals and a space), but this is exactly what the canonical table contains for that song, so it matches.

**A problem:** the Wu-Tang line is not in the canonical table. MusicBrainz has the song as just `C.R.E.A.M.`, whose key is `wutangclancream`. The same happens with `(Remastered 2011)`, `- Live`, `feat. ...` and so on.

### The cleaning rules (a second key)

We make a **second key** from a cleaned version of the song:

- **R1** - remove everything in brackets `( )` or `[ ]` from the title (`Song (Remastered 2011)` becomes `Song`)
- **R2** - cut the title at the first ` - ` (`Song - Live` becomes `Song`)
- **R3** - cut `feat.` / `ft.` / `featuring` and everything after it, in both artist and title

Below, the plain key is tried first. The cleaned key is only used for songs that did not match, so the rules can never replace a correct match.

In [152]:
def clean_title(title):
    title = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", title)                        # R1: drop brackets
    title = title.split(" - ")[0]                                              # R2: cut at " - "
    title = re.sub(r"\s+(feat\.?|ft\.?|featuring)\s.*$", "", title, flags=re.I)   # R3: cut feat.
    return title.strip()

def clean_artist(artist):
    return re.sub(r"\s+(feat\.?|ft\.?|featuring)\s.*$", "", artist, flags=re.I).strip()   # R3 only

def make_clean_key(artist, title):
    t = clean_title(title)
    if not t:                      # if cleaning leaves nothing, keep the original title
        t = title
    return make_key(clean_artist(artist), t)

tests = [
    ("Wu-Tang Clan", "C.R.E.A.M. (Cash Rules Everything Around Me)"),
    ("The Beatles", "Help! - Remastered 2009"),
    ("deadmau5 feat. Gerard Way", "Professional Griefers (vocal mix)"),
    ("Queen", "Bohemian Rhapsody"),
]
pd.DataFrame([(a, t, make_key(a, t), make_clean_key(a, t)) for a, t in tests], columns=["artist", "title", "plain key", "clean key"])

,artist,title,plain key,clean key
0,Wu-Tang Clan,C.R.E.A.M. (Cash Rules Everything Around Me),wutangclancreamcashruleseverythingaroundme,wutangclancream
1,The Beatles,Help! - Remastered 2009,thebeatleshelpremastered2009,thebeatleshelp
2,deadmau5 feat. Gerard Way,Professional Griefers (vocal mix),deadmau5featgerardwayprofessionalgriefersvocalmix,deadmau5professionalgriefers
3,Queen,Bohemian Rhapsody,queenbohemianrhapsody,queenbohemianrhapsody


### Collaborations: two more keys

MusicBrainz often writes a collaboration as **one artist credit**, such as `Gorillaz feat. De La Soul` or `Muse & Ellie Goulding`, while ListenBrainz may have `Gorillaz, De La Soul`, or just `Gorillaz` with `(feat. De La Soul)` in the title. A key drops punctuation, so `,` and `&` are not a problem, but the word `feat` is. We add two more keys for the songs where this happens:

- **collab key** - an artist field with several artists (separated by `,` `;` `&` `and` `x`) is rewritten as `first feat the others`.
- **title-feat key** - if the title has `(feat. X)` or `(with X)` in brackets, X is moved to the artist: `Gorillaz` + `The Hardest Thing (feat. Tony Allen)` becomes `Gorillaz feat Tony Allen` + `The Hardest Thing`.

Other joining words (`vs`, `with`, `x` between artists) exist, but together they are well under 0.2% of the listens, so we leave them out.

In [153]:
ARTIST_SPLIT = re.compile(r"\s*(?:;|,|&|\band\b|\bx\b)\s*", re.I)
FEAT_IN_TITLE = re.compile(r"[\(\[]\s*(?:feat\.?|ft\.?|featuring|with)\s+([^\)\]]+)[\)\]]", re.I)

def make_collab_key(artist, title):
    """Key for 'A, B' -> 'A feat B'. Empty if the artist field has only one artist."""
    parts = [p for p in ARTIST_SPLIT.split(clean_artist(artist)) if p.strip()]
    if len(parts) < 2:
        return ""
    return make_key(parts[0] + " feat " + " ".join(parts[1:]), clean_title(title) or title)

def make_title_feat_key(artist, title):
    """Key for 'A' + 'Song (feat. B)' -> 'A feat B' + 'Song'. Empty if the title has no feat. part."""
    m = FEAT_IN_TITLE.search(title)
    if not m:
        return ""
    names = [p for p in ARTIST_SPLIT.split(m.group(1)) if p.strip()]
    return make_key(clean_artist(artist) + " feat " + " ".join(names), clean_title(title) or title)

print(make_collab_key("Gorillaz, De La Soul", "Feel Good Inc."))
print(make_title_feat_key("Gorillaz", "The Hardest Thing (feat. Tony Allen)"))

gorillazfeatdelasoulfeelgoodinc
gorillazfeattonyallenthehardestthing


Now apply all four keys to all distinct songs (once per song, not once per listen). This takes a minute or two.

In [154]:
songs["key_plain"] = [make_key(a, t) for a, t in zip(songs["artist_name"], songs["track_name"])]
songs["key_clean"] = [make_clean_key(a, t) for a, t in zip(songs["artist_name"], songs["track_name"])]
songs["key_collab"] = [make_collab_key(a, t) for a, t in zip(songs["artist_name"], songs["track_name"])]
songs["key_title_feat"] = [make_title_feat_key(a, t) for a, t in zip(songs["artist_name"], songs["track_name"])]

# A song with an empty artist or an empty title cannot be looked up: no key.
no_name = (songs["artist_name"].str.strip() == "") | (songs["track_name"].str.strip() == "")
songs.loc[no_name, ["key_plain", "key_clean", "key_collab", "key_title_feat"]] = ""

has_key = songs["key_plain"] != ""
changed = has_key & (songs["key_plain"] != songs["key_clean"])
print(f"songs without a usable key:          {no_name.sum():,}")
print(f"songs with a collab key: {(songs['key_collab'] != '').sum():,}, with a title-feat key: {(songs['key_title_feat'] != '').sum():,}")
print(f"songs where the cleaned key differs: {changed.sum():,} ({changed.sum() / has_key.sum() * 100:.1f}% of songs), {songs.loc[changed, 'n_listens'].sum() / songs['n_listens'].sum() * 100:.1f}% of listens")
songs[changed].sort_values("n_listens", ascending=False).head(10)[["artist_name", "track_name", "key_plain", "key_clean"]]

songs without a usable key:          1
songs with a collab key: 69,780, with a title-feat key: 22,477
songs where the cleaned key differs: 213,682 (20.1% of songs), 13.4% of listens


,artist_name,track_name,key_plain,key_clean
218053,Daft Punk,Get Lucky (feat. Pharrell Williams),daftpunkgetluckyfeatpharrellwilliams,daftpunkgetlucky
323423,Feisty 2 Gunns,"Empty Whitt Out You (Feisty 2 Gunns, Chino Gra...",feisty2gunnsemptywhittoutyoufeisty2gunnschinog...,feisty2gunnsemptywhittoutyou
1047645,Macklemore & Ryan Lewis,Thrift Shop (feat. Wanz),macklemoreryanlewisthriftshopfeatwanz,macklemoreryanlewisthriftshop
866269,Daft Punk,Instant Crush (feat. Julian Casablancas),daftpunkinstantcrushfeatjuliancasablancas,daftpunkinstantcrush
550878,Daft Punk,Lose Yourself to Dance (feat. Pharrell Williams),daftpunkloseyourselftodancefeatpharrellwilliams,daftpunkloseyourselftodance
149651,Macklemore & Ryan Lewis,Can't Hold Us (feat. Ray Dalton),macklemoreryanlewiscantholdusfeatraydalton,macklemoreryanlewiscantholdus
267765,Daft Punk,Touch (feat. Paul Williams),daftpunktouchfeatpaulwilliams,daftpunktouch
17769,"Calvin Harris, Ellie Goulding",I Need Your Love (feat. Ellie Goulding),calvinharriselliegouldingineedyourlovefeatelli...,calvinharriselliegouldingineedyourlove
914559,Icona Pop,I Love It (feat. Charli XCX),iconapopiloveitfeatcharlixcx,iconapopiloveit
583766,Daft Punk,Doin' It Right (feat. Panda Bear),daftpunkdoinitrightfeatpandabear,daftpunkdoinitright


## 4. The MusicBrainz canonical table

The canonical table is a 7.7 GB csv inside a compressed archive (`.tar.zst`). We never unpack it on disk - we read it as a **stream**, block by block, and keep only the rows whose `combined_lookup` is one of our keys. This takes about a minute.

First, the first 5 rows, to see what it looks like:

In [155]:
MEMBER = "canonical/canonical_musicbrainz_data.csv"       # the csv we want, inside the archive

class Stream(io.RawIOBase):
    """A small wrapper that makes the stream from the archive look like a normal file that can only be read forward."""
    def __init__(self, f):
        self.f = f
    def readable(self):
        return True
    def readinto(self, b):
        data = self.f.read(len(b))
        b[:len(data)] = data
        return len(data)

def open_canonical():
    """Opens the dump and returns a stream of the canonical csv."""
    reader = zstandard.ZstdDecompressor().stream_reader(open(CANON, "rb"))     # undo the zstd compression while reading
    tar = tarfile.open(fileobj=reader, mode="r|")                                # read the tar archive as a stream
    for member in tar:
        if member.name.endswith(MEMBER):
            return io.BufferedReader(Stream(tar.extractfile(member)), buffer_size=16 * 1024 * 1024)
    raise ValueError("canonical csv not found in the archive")

pd.read_csv(open_canonical(), nrows=5)

,id,artist_credit_id,artist_mbids,artist_credit_name,release_mbid,release_name,recording_mbid,recording_name,combined_lookup,score
0,1,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,61efbf58-5be4-40ba-9bef-49c92cc5b8ca,Sofé Linzy,7b1f193b-c9ba-48d6-bd6f-8afd02d489a2,Pot Pourri Sega,variousartistspotpourrisega,189058
1,2,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,82b37a32-e983-4b6b-b353-f38b9018ea7f,滾石民歌時代百大經典,4cbd4a88-eff8-47e5-8d18-683b1cdff891,乡愁四韵,variousartistsXiang Chou Si Yun,191441
2,3,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,82b37a32-e983-4b6b-b353-f38b9018ea7f,滾石民歌時代百大經典,77c52b21-0ce0-4c49-a99d-d25ebd4f5f71,白霏霏,variousartistsBai Fei Fei,191441
3,4,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,82b37a32-e983-4b6b-b353-f38b9018ea7f,滾石民歌時代百大經典,9eacbc95-da23-43b1-aa80-2d27c64e1070,生日歌,variousartistsSheng Ri Ge,191441
4,5,1,89ad4ac3-39f7-470e-963a-56509c546377,Various Artists,82b37a32-e983-4b6b-b353-f38b9018ea7f,滾石民歌時代百大經典,b1abb565-8cee-412c-af11-b5602af79760,民歌,variousartistsMin Ge,191441


What the columns mean:

- `artist_credit_name`, `recording_name`: the artist and the song title as MusicBrainz writes them.
- `combined_lookup`: artist + title as a key. **This is the column we compare with.**
- `recording_mbid`, `artist_mbids`, `release_mbid`: the MusicBrainz ids we want.
- `score`: a priority number. If several rows have the same key, the row with the lowest score is preferred.

Now the stream. We read 32 MB at a time, and keep the rows with one of our keys.

In [156]:
keys = set(songs["key_plain"]) | set(songs["key_clean"]) | set(songs["key_collab"]) | set(songs["key_title_feat"])
keys.discard("")                                  # an empty key means: no usable key
print(f"{len(keys):,} different keys to look for")

columns = ["artist_mbids", "release_mbid", "recording_mbid", "combined_lookup", "score"]          # only what we need
all_columns = ["id", "artist_credit_id", "artist_mbids", "artist_credit_name", "release_mbid", "release_name",
               "recording_mbid", "recording_name", "combined_lookup", "score"]

value_set = pa.array(list(keys))                  # our keys, in a format that is fast to test against
del keys
csv_reader = pacsv.open_csv(
    open_canonical(),
    read_options=pacsv.ReadOptions(block_size=32 * 1024 * 1024),
    convert_options=pacsv.ConvertOptions(column_types={c: pa.string() for c in all_columns}, include_columns=columns),
)

found = []
rows_seen = 0
start = time.time()
for number, block in enumerate(csv_reader, start=1):
    rows_seen += block.num_rows
    found.append(block.filter(pc.is_in(block["combined_lookup"], value_set=value_set)))     # only rows with one of our keys
    if number % 20 == 0:
        print(f"block {number:3d}: {rows_seen:>12,} rows read | {sum(t.num_rows for t in found):>9,} kept | {time.time() - start:4.0f} s")

canon_hits = pa.Table.from_batches(found)
print(f"\ndone: {rows_seen:,} rows in the canonical table, {canon_hits.num_rows:,} of them have one of our keys ({time.time() - start:.0f} s)")

# free the memory we no longer need
del found, csv_reader, value_set
gc.collect()
pa.default_memory_pool().release_unused()

1,199,238 different keys to look for
block  20:    3,005,197 rows read |   258,356 kept |    4 s
block  40:    6,029,715 rows read |   380,550 kept |    7 s
block  60:    9,083,539 rows read |   481,137 kept |   11 s
block  80:   12,142,654 rows read |   579,466 kept |   14 s
block 100:   14,863,409 rows read |   661,877 kept |   18 s
block 120:   17,623,457 rows read |   693,926 kept |   22 s
block 140:   20,268,755 rows read |   708,673 kept |   25 s
block 160:   23,000,672 rows read |   717,652 kept |   29 s
block 180:   25,711,276 rows read |   726,067 kept |   32 s
block 200:   28,446,644 rows read |   731,586 kept |   35 s
block 220:   31,071,807 rows read |   735,964 kept |   39 s

done: 32,442,553 rows in the canonical table, 737,978 of them have one of our keys (40 s)


## 5. Link the songs to the MusicBrainz ids

Now we have two tables inside DuckDB: our `songs` and the matching canonical rows `canon_hits`. A SQL `JOIN` links them:

- first try the **plain key** (`matched_by = plain`)
- if that finds nothing, try the **cleaned key** (`clean`), then the **collab key** (`collab`), then the **title-feat key** (`title_feat`), otherwise `none`
- if ListenBrainz already gave us an id, we trust that over our own lookup (`id_source`)
- a song can have several artists in MusicBrainz: we take the first one
- if several canonical rows share a key, we keep the one with the lowest score

In [157]:
con.register("songs", songs)
con.register("canon_hits", canon_hits)

run("""
    CREATE OR REPLACE TABLE linked AS
    WITH canon AS (
        SELECT combined_lookup, recording_mbid, release_mbid,
               nullif(split_part(regexp_replace(artist_mbids, '[{}" ]', '', 'g'), ',', 1), '') AS artist_mbid
        FROM canon_hits
        QUALIFY row_number() OVER (PARTITION BY combined_lookup ORDER BY score::BIGINT) = 1
    )
    SELECT s.artist_name, s.track_name, s.n_listens,
           CASE WHEN p.recording_mbid IS NOT NULL THEN 'plain'
                WHEN c.recording_mbid IS NOT NULL THEN 'clean'
                WHEN d.recording_mbid IS NOT NULL THEN 'collab'
                WHEN f.recording_mbid IS NOT NULL THEN 'title_feat'
                ELSE 'none' END                                                           AS matched_by,
           CASE WHEN s.lb_recording_mbid <> '' THEN 'listenbrainz'
                WHEN coalesce(p.recording_mbid, c.recording_mbid, d.recording_mbid, f.recording_mbid) IS NOT NULL THEN 'canonical'
                ELSE 'none' END                                                           AS id_source,
           coalesce(nullif(s.lb_recording_mbid, ''), p.recording_mbid, c.recording_mbid, d.recording_mbid, f.recording_mbid) AS recording_mbid,
           coalesce(nullif(s.lb_artist_mbid, ''),    p.artist_mbid,    c.artist_mbid,    d.artist_mbid,    f.artist_mbid)    AS artist_mbid,
           coalesce(nullif(s.lb_release_mbid, ''),   p.release_mbid,   c.release_mbid,   d.release_mbid,   f.release_mbid)   AS release_mbid
    FROM songs s
    LEFT JOIN canon p ON p.combined_lookup = s.key_plain AND s.key_plain <> ''
    LEFT JOIN canon c ON c.combined_lookup = s.key_clean AND s.key_clean <> ''
    LEFT JOIN canon d ON d.combined_lookup = s.key_collab AND s.key_collab <> ''
    LEFT JOIN canon f ON f.combined_lookup = s.key_title_feat AND s.key_title_feat <> ''
""")

con.sql("""
    SELECT matched_by, count(*) AS songs, sum(n_listens) AS listens,
           round(100 * sum(n_listens) / (SELECT sum(n_listens) FROM linked), 1) AS pct_of_listens
    FROM linked GROUP BY matched_by ORDER BY listens DESC
""").df()

,matched_by,songs,listens,pct_of_listens
0,plain,734392,4115071.0,82.5
1,none,251133,611427.0,12.3
2,clean,68506,203204.0,4.1
3,title_feat,7545,44014.0,0.9
4,collab,2753,15197.0,0.3


The songs with `none` could not be found in MusicBrainz. The most-played ones show what is missing:

In [158]:
con.sql("""
    SELECT artist_name, track_name, n_listens FROM linked
    WHERE recording_mbid IS NULL ORDER BY n_listens DESC LIMIT 12
""").df()

,artist_name,track_name,n_listens
0,Feisty 2 Gunns,"Empty Whitt Out You (Feisty 2 Gunns, Chino Gra...",1116
1,"Macklemore & Ryan Lewis, Macklemore, Ryan Lewi...",Thrift Shop (feat. Wanz),550
2,"Macklemore & Ryan Lewis, Macklemore, Ryan Lewi...",Can't Hold Us (feat. Ray Dalton),433
3,Avicii,Avicii - Wake Me Up (Radio Edit),406
4,Lorentz & Sakarias,Molnen (Ni kan inte nå mig),399
5,Afromania,Minna no Peace,371
6,Marco & Snickin,Eigene Wege,342
7,巡音ルカ,Circus Monster,300
8,Anna Yvette,Open Window - Original Mix,293
9,Hiroyuki Sawano,The Reluctant Heroes (vocal: mpi),243


## 6. Find the genres

MusicBrainz users can tag songs and artists. A tag counts as a **genre** only if its name is on MusicBrainz's official genre list (about 2,200 names). For every song we take:

1. the **most-voted genre of the recording** (the song), and if it has none
2. the **most-voted genre of the artist**.

Ties are decided alphabetically. The MusicBrainz tables are big tab-separated text files (the recording table is 4.6 GB); DuckDB reads them directly and keeps only what we need. This takes a few minutes.

First a small helper that tells DuckDB how to read one of these files:

In [159]:
def tsv(name, columns):
    """SQL for reading one MusicBrainz table (tab-separated, no header, a backslash-N marks a missing value)."""
    cols = ", ".join(f"'{c}': 'VARCHAR'" for c in columns)
    return (f"read_csv('{MB / name}', delim='\\t', header=false, quote='', escape='', nullstr='\\N', "
            f"columns={{{cols}}}, auto_detect=false, strict_mode=false, ignore_errors=true)")

# the official genre list, and the tags that are on it
run(f"CREATE OR REPLACE TABLE genre AS SELECT name FROM {tsv('genre', ['id', 'gid', 'name'])}")
run(f"""CREATE OR REPLACE TABLE genre_tag AS
                SELECT t.id::BIGINT AS tag_id, t.name AS genre
                FROM {tsv('tag', ['id', 'name'])} t WHERE t.name IN (SELECT name FROM genre)""")
print(scalar("SELECT count(*) AS official_genres FROM genre"), "official genres,",
      scalar("SELECT count(*) FROM genre_tag"), "of them exist as a tag")

2221 official genres, 1924 of them exist as a tag


### 6a. The genre of each recording

In [160]:
start = time.time()
# the numeric id of every recording we need (the tag tables use the number, not the mbid)
run(f"""CREATE OR REPLACE TABLE rec AS
                SELECT id::BIGINT AS id, gid FROM {tsv('recording', ['id', 'gid', 'name', 'ac', 'length'])}
                WHERE gid IN (SELECT recording_mbid FROM linked WHERE recording_mbid IS NOT NULL)""")
print(f"{scalar('SELECT count(*) FROM rec'):,} recordings found in MusicBrainz ({time.time() - start:.0f} s)")

# tags of those recordings that are official genres, with votes > 0; the best one per recording wins
run(f"""CREATE OR REPLACE TABLE rec_genre AS
                SELECT r.gid, g.genre
                FROM {tsv('recording_tag', ['recording', 'tag', 'count'])} x
                JOIN rec r ON r.id = x.recording::BIGINT
                JOIN genre_tag g ON g.tag_id = x.tag::BIGINT
                WHERE x.count::INT > 0
                QUALIFY row_number() OVER (PARTITION BY r.gid ORDER BY x.count::INT DESC, g.genre) = 1""")
print(f"{scalar('SELECT count(*) FROM rec_genre'):,} recordings have a genre ({time.time() - start:.0f} s)")

606,621 recordings found in MusicBrainz (6 s)
243,374 recordings have a genre (7 s)


### 6b. The genre of each artist (for the songs where the recording has none)

In [161]:
start = time.time()
artist_columns = ['id', 'gid', 'name', 'sort_name', 'begin_y', 'begin_m', 'begin_d', 'end_y', 'end_m', 'end_d', 'type', 'area', 'gender', 'comment']
run(f"""CREATE OR REPLACE TABLE art AS
                SELECT id::BIGINT AS id, gid FROM {tsv('artist', artist_columns)}
                WHERE gid IN (SELECT artist_mbid FROM linked WHERE artist_mbid IS NOT NULL)""")
run(f"""CREATE OR REPLACE TABLE artist_genre AS
                SELECT a.gid, g.genre
                FROM {tsv('artist_tag', ['artist', 'tag', 'count'])} x
                JOIN art a ON a.id = x.artist::BIGINT
                JOIN genre_tag g ON g.tag_id = x.tag::BIGINT
                WHERE x.count::INT > 0
                QUALIFY row_number() OVER (PARTITION BY a.gid ORDER BY x.count::INT DESC, g.genre) = 1""")
print(f"{scalar('SELECT count(*) FROM art'):,} artists found, {scalar('SELECT count(*) FROM artist_genre'):,} have a genre ({time.time() - start:.0f} s)")

78,341 artists found, 32,951 have a genre (0 s)


### 6c. Put it together: one genre per song

In [162]:
run("""
    CREATE OR REPLACE TABLE song_genres AS
    SELECT l.*,
           coalesce(rg.genre, ag.genre) AS genre,
           CASE WHEN rg.genre IS NOT NULL THEN 'recording'
                WHEN ag.genre IS NOT NULL THEN 'artist'
                ELSE 'none' END         AS genre_source
    FROM linked l
    LEFT JOIN rec_genre rg ON rg.gid = l.recording_mbid
    LEFT JOIN artist_genre ag ON ag.gid = l.artist_mbid
""")
con.sql("""
    SELECT genre_source, count(*) AS songs, sum(n_listens) AS listens,
           round(100 * sum(n_listens) / (SELECT sum(n_listens) FROM song_genres), 1) AS pct_of_listens
    FROM song_genres GROUP BY genre_source ORDER BY listens DESC
""").df()

,genre_source,songs,listens,pct_of_listens
0,artist,360802,1972877.0,39.5
1,recording,283910,1948986.0,39.1
2,none,419617,1067050.0,21.4


## 7. Save

- **The song table** as a parquet file: one row per song with ids and genre. It is big and only an intermediate step.
- **The genre counts** as a csv in `Output_dev/genre_counts/<year>/`: how many listens each genre got. This is what the analysis and plots start from.
- **The coverage** as a csv: how many listens could be linked, and how many could not (and why).

In [163]:
song_file = PARQ / f"songs_genre_{YEAR}_{SUFFIX}.parquet"
run(f"COPY song_genres TO '{song_file}' (FORMAT parquet)")
print(f"saved the song table: {song_file.name} ({song_file.stat().st_size / 1e6:,.0f} MB)")

genre_counts = con.sql(f"""
    SELECT {YEAR} AS year, genre, sum(n_listens)::BIGINT AS listens, count(*) AS songs
    FROM song_genres WHERE genre IS NOT NULL
    GROUP BY genre ORDER BY listens DESC
""").df()
genre_counts.to_csv(CSV_DIR / f"genre_counts_{YEAR}_{SUFFIX}.csv", index=False)

coverage = con.sql(f"""
    SELECT {YEAR} AS year,
           CASE WHEN genre_source = 'recording' THEN '1 genre from the recording'
                WHEN genre_source = 'artist'    THEN '2 genre from the artist'
                WHEN recording_mbid IS NOT NULL OR artist_mbid IS NOT NULL THEN '3 found in MusicBrainz, but no genre'
                ELSE '4 not found in MusicBrainz' END AS category,
           count(*) AS songs, sum(n_listens)::BIGINT AS listens
    FROM song_genres GROUP BY category ORDER BY category
""").df()
coverage.to_csv(CSV_DIR / f"coverage_{YEAR}_{SUFFIX}.csv", index=False)

print(f"saved genre_counts_{YEAR}_{SUFFIX}.csv ({len(genre_counts):,} genres) and coverage_{YEAR}_{SUFFIX}.csv in {CSV_DIR}")
coverage.assign(pct_of_listens=(coverage["listens"] / coverage["listens"].sum() * 100).round(1))

saved the song table: songs_genre_2013_dev.parquet (120 MB)
saved genre_counts_2013_dev.csv (991 genres) and coverage_2013_dev.csv in /Users/hanshornbek/Documents/ITU/3. Semester/Data Mining/DAMML-group -project/Assignment 02/Hans/Part_01 (Genre Popularity)/Output/genre_counts/2013


,year,category,songs,listens,pct_of_listens
0,2013,1 genre from the recording,283910,1948986,39.1
1,2013,2 genre from the artist,360802,1972877,39.5
2,2013,"3 found in MusicBrainz, but no genre",178956,505579,10.1
3,2013,4 not found in MusicBrainz,240661,561471,11.3


In [164]:
genre_counts.head(10)

,year,genre,listens,songs
0,2013,electronic,319805,39307
1,2013,alternative rock,294549,33456
2,2013,rock,288022,43546
3,2013,indie rock,234726,19422
4,2013,pop,220469,28093
5,2013,hip hop,201487,34714
6,2013,ambient,80684,15676
7,2013,hard rock,67524,11554
8,2013,folk,63787,13124
9,2013,heavy metal,61721,10907


## What to notice

- The match key and the cleaning rules are the only Python-specific logic. Everything else is SQL on the data.
- **The genre belongs to the song** (recording), with the artist's genre as a fallback. A song with no tags on either side has no genre.
- `coverage` tells how much of the listening we can say something about. The genre questions are about the listens that have a genre.
- **To scale up:** change `YEAR`, or set `DEV_ONLY = False` for all users. The next notebook plots the genre counts.